In [0]:
from pyspark.sql import functions as F
from pii_utils import PiiCrypto

container_name = "data"
storage_account_name = "stdevnortheuropebfn0"

base_path = (
    f"abfss://{container_name}@"
    f"{storage_account_name}.dfs.core.windows.net"
)

# Source paths
yellow_source_path = f"{base_path}/landing/yellow"
green_source_path = f"{base_path}/landing/green"
zones_source_path = f"{base_path}/reference/taxi_zone_lookup.csv"


# Auto Loader schema paths
yellow_schema_path = f"{base_path}/schemas/bronze/yellow"
green_schema_path = f"{base_path}/schemas/bronze/green"


# Streaming checkpoint paths
yellow_checkpoint_path = f"{base_path}/checkpoints/bronze/yellow"
green_checkpoint_path = f"{base_path}/checkpoints/bronze/green"


# Read Yellow and Green taxi files incrementally
yellow_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", yellow_schema_path)
    .load(yellow_source_path)
)

green_raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", green_schema_path)
    .load(green_source_path)
)


In [0]:
# Get the AES key from Databricks Secrets
pii_key = dbutils.secrets.get(
    scope="pii_scope",
    key="pii_aes_key"
)

crypto = PiiCrypto(encryption_key=pii_key)

pii_columns = [
    "customer_first_name",
    "customer_last_name",
    "customer_email"
]


# Encrypt PII and create Bronze partition columns
def prepare_bronze(df, pickup_column):
    partitioned_df = (
        df
        .withColumn("year", F.year(F.col(pickup_column)))
        .withColumn("month", F.month(F.col(pickup_column)))
        .withColumn("day", F.dayofmonth(F.col(pickup_column)))
    )

    return crypto.encrypt_columns(
        partitioned_df,
        pii_columns
    )


yellow_bronze_df = prepare_bronze(
    yellow_raw_df,
    "tpep_pickup_datetime"
)

green_bronze_df = prepare_bronze(
    green_raw_df,
    "lpep_pickup_datetime"
)


In [0]:
# Load the static taxi-zone reference table
taxi_zones_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zones_source_path)
)

(
    taxi_zones_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.taxi_zones_raw")
)

In [0]:
# Start the Yellow Taxi stream
yellow_query = (
    yellow_bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", yellow_checkpoint_path)
    .partitionBy("year", "month", "day")
    .trigger(processingTime="20 seconds")
    .queryName("bronze_yellow_taxi_ingestion")
    .toTable("bronze.yellow_trips_raw")
)

display(spark.readStream.table("bronze.yellow_trips_raw"))

In [0]:
# Start the Green Taxi stream
green_query = (
    green_bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", green_checkpoint_path)
    .partitionBy("year", "month", "day")
    .trigger(processingTime="20 seconds")
    .queryName("bronze_green_taxi_ingestion")
    .toTable("bronze.green_trips_raw")
)

display(spark.readStream.table("bronze.green_trips_raw"))